### Scenario: E-commerce Order Support Label Engineering

You are working as an AI engineer preparing e-commerce e-commerce order support data for a future model-training or fine-tuning workflow. The raw data contains customer messages, issue types, support responses, order references, tracking codes, and product SKUs. These fields need to be transformed into consistent, structured training examples.

In this notebook, you will:
- map raw issue types to standardized multi-class labels;
- convert those labels into numeric IDs;
- create instruction-response and prompt-target examples for supervised fine-tuning;
- label order references, tracking codes, and product SKUs with BIO-style sequence tags;
- export the resulting datasets for reuse.

### Learning Objectives

- Explain why raw business text needs consistent labels before model training.
- Engineer standardized classes from e-commerce support issue types.
- Build a reproducible label-to-ID mapping.
- Prepare instruction-response examples and prompt-target pairs.
- Apply BIO tags to entity mentions in text.
- Export the transformed datasets to CSV and JSONL files.



### Import Libraries

In [ ]:
import pandas as pd

### Create Raw E-commerce Order Support Data

In [2]:
raw_data = pd.DataFrame({
    "customer_message": [
        "Where is my order ORD-1042?",
        "I received the wrong item for order ORD-1088.",
        "My card was charged twice for order ORD-1103.",
        "The payment for order ORD-1120 keeps failing.",
        "My package for order ORD-1151 is delayed.",
        "The tracking page for order ORD-1164 has not updated.",
        "I want to return the shoes from order ORD-1190.",
        "When will I receive a refund for order ORD-1205?",
        "The tracking code TRK-78421 says delivered, but I have no package.",
        "Product SKU-AX19 arrived damaged in order ORD-1234."
    ],
    "issue_type": [
        "order_status",
        "order_problem",
        "payment",
        "payment",
        "delivery",
        "delivery",
        "return",
        "refund",
        "delivery",
        "return"
    ],
    "support_response": [
        "You can view the latest status for ORD-1042 in your order history.",
        "We are sorry about the incorrect item. We can help arrange a replacement.",
        "We will review the duplicate charge and follow up with you.",
        "Please verify your payment details or try another payment method.",
        "We apologize for the delay. Please check the carrier updates for ORD-1151.",
        "We will check the tracking status and provide an update.",
        "You can start a return from the order details page.",
        "We will check the refund status for ORD-1205.",
        "We are sorry the package is missing. We will investigate with the carrier.",
        "We are sorry the item arrived damaged. We can help arrange a return or replacement."
    ]
})

raw_data


,customer_message,issue_type,support_response
0,Where is my order ORD-1042?,order_status,You can view the latest status for ORD-1042 in...
1,I received the wrong item for order ORD-1088.,order_problem,We are sorry about the incorrect item. We can ...
2,My card was charged twice for order ORD-1103.,payment,We will review the duplicate charge and follow...
3,The payment for order ORD-1120 keeps failing.,payment,Please verify your payment details or try anot...
4,My package for order ORD-1151 is delayed.,delivery,We apologize for the delay. Please check the c...
5,The tracking page for order ORD-1164 has not u...,delivery,We will check the tracking status and provide ...
6,I want to return the shoes from order ORD-1190.,return,You can start a return from the order details ...
7,When will I receive a refund for order ORD-1205?,refund,We will check the refund status for ORD-1205.
8,"The tracking code TRK-78421 says delivered, bu...",delivery,We are sorry the package is missing. We will i...
9,Product SKU-AX19 arrived damaged in order ORD-...,return,We are sorry the item arrived damaged. We can ...


### Engineer Multi-Class Labels

In [ ]:
label_mapping = {
    "order_status": "ORDER_ISSUE",
    "order_problem": "ORDER_ISSUE",
    "payment": "PAYMENT_ISSUE",
    "delivery": "DELIVERY_ISSUE",
    "return": "RETURN_ISSUE",
    "refund": "REFUND_ISSUE"
}

raw_data["engineered_label"] = raw_data["issue_type"].map(label_mapping)


assert raw_data["engineered_label"].notna().all(), "An issue_type has no label mapping."

raw_data[["customer_message", "issue_type", "engineered_label"]]


,customer_message,issue_type,engineered_label
0,Where is my order ORD-1042?,order_status,ORDER_ISSUE
1,I received the wrong item for order ORD-1088.,order_problem,ORDER_ISSUE
2,My card was charged twice for order ORD-1103.,payment,PAYMENT_ISSUE
3,The payment for order ORD-1120 keeps failing.,payment,PAYMENT_ISSUE
4,My package for order ORD-1151 is delayed.,delivery,DELIVERY_ISSUE
5,The tracking page for order ORD-1164 has not u...,delivery,DELIVERY_ISSUE
6,I want to return the shoes from order ORD-1190.,return,RETURN_ISSUE
7,When will I receive a refund for order ORD-1205?,refund,REFUND_ISSUE
8,"The tracking code TRK-78421 says delivered, bu...",delivery,DELIVERY_ISSUE
9,Product SKU-AX19 arrived damaged in order ORD-...,return,RETURN_ISSUE


### Convert Labels into Numeric IDs

In [4]:
unique_labels = sorted(raw_data["engineered_label"].unique())

label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

raw_data["label_id"] = raw_data["engineered_label"].map(label_to_id)

print("Label to ID Mapping:")
print(label_to_id)

raw_data[["customer_message", "engineered_label", "label_id"]]

Label to ID Mapping:
{'DELIVERY_ISSUE': 0, 'ORDER_ISSUE': 1, 'PAYMENT_ISSUE': 2, 'REFUND_ISSUE': 3, 'RETURN_ISSUE': 4}


,customer_message,engineered_label,label_id
0,Where is my order ORD-1042?,ORDER_ISSUE,1
1,I received the wrong item for order ORD-1088.,ORDER_ISSUE,1
2,My card was charged twice for order ORD-1103.,PAYMENT_ISSUE,2
3,The payment for order ORD-1120 keeps failing.,PAYMENT_ISSUE,2
4,My package for order ORD-1151 is delayed.,DELIVERY_ISSUE,0
5,The tracking page for order ORD-1164 has not u...,DELIVERY_ISSUE,0
6,I want to return the shoes from order ORD-1190.,RETURN_ISSUE,4
7,When will I receive a refund for order ORD-1205?,REFUND_ISSUE,3
8,"The tracking code TRK-78421 says delivered, bu...",DELIVERY_ISSUE,0
9,Product SKU-AX19 arrived damaged in order ORD-...,RETURN_ISSUE,4


### Create Instruction-Response Pairs

In [5]:
instruction_data = pd.DataFrame()

instruction_data["instruction"] = "Classify the customer message into the correct support category."
instruction_data["input"] = raw_data["customer_message"]
instruction_data["response"] = raw_data["engineered_label"]

instruction_data

,instruction,input,response
0,NaN,Where is my order ORD-1042?,ORDER_ISSUE
1,NaN,I received the wrong item for order ORD-1088.,ORDER_ISSUE
2,NaN,My card was charged twice for order ORD-1103.,PAYMENT_ISSUE
3,NaN,The payment for order ORD-1120 keeps failing.,PAYMENT_ISSUE
4,NaN,My package for order ORD-1151 is delayed.,DELIVERY_ISSUE
5,NaN,The tracking page for order ORD-1164 has not u...,DELIVERY_ISSUE
6,NaN,I want to return the shoes from order ORD-1190.,RETURN_ISSUE
7,NaN,When will I receive a refund for order ORD-1205?,REFUND_ISSUE
8,NaN,"The tracking code TRK-78421 says delivered, bu...",DELIVERY_ISSUE
9,NaN,Product SKU-AX19 arrived damaged in order ORD-...,RETURN_ISSUE


### Build Prompt-Target Format

In [6]:
def build_prompt(customer_message):
    return f"""Instruction:
Classify the e-commerce customer message into one of these labels:
ORDER_STATUS, ORDER_PROBLEM, PAYMENT_ISSUE, DELIVERY_ISSUE, RETURN_REQUEST, REFUND_REQUEST

Customer Message:
{customer_message}

Label:"""

instruction_data["prompt"] = instruction_data["input"].apply(build_prompt)
instruction_data["target"] = instruction_data["response"]

instruction_data[["prompt", "target"]]


,prompt,target
0,Instruction:\nClassify the e-commerce customer...,ORDER_ISSUE
1,Instruction:\nClassify the e-commerce customer...,ORDER_ISSUE
2,Instruction:\nClassify the e-commerce customer...,PAYMENT_ISSUE
3,Instruction:\nClassify the e-commerce customer...,PAYMENT_ISSUE
4,Instruction:\nClassify the e-commerce customer...,DELIVERY_ISSUE
5,Instruction:\nClassify the e-commerce customer...,DELIVERY_ISSUE
6,Instruction:\nClassify the e-commerce customer...,RETURN_ISSUE
7,Instruction:\nClassify the e-commerce customer...,REFUND_ISSUE
8,Instruction:\nClassify the e-commerce customer...,DELIVERY_ISSUE
9,Instruction:\nClassify the e-commerce customer...,RETURN_ISSUE


### Create Sequence Tagging Data

In [7]:
entity_data = pd.DataFrame({
    "text": [
        "Please check order ORD-1042 for me.",
        "The carrier tracking code TRK-78421 has not changed.",
        "Product SKU-AX19 arrived damaged.",
        "I need help with order ORD-1234 and tracking code TRK-99102."
    ],
    "entity_value": [
        "ORD-1042",
        "TRK-78421",
        "SKU-AX19",
        "ORD-1234"
    ],
    "entity_type": [
        "ORDER_ID",
        "TRACKING_CODE",
        "PRODUCT_SKU",
        "ORDER_ID"
    ]
})

entity_data


,text,entity_value,entity_type
0,Please check order ORD-1042 for me.,ORD-1042,ORDER_ID
1,The carrier tracking code TRK-78421 has not ch...,TRK-78421,TRACKING_CODE
2,Product SKU-AX19 arrived damaged.,SKU-AX19,PRODUCT_SKU
3,I need help with order ORD-1234 and tracking c...,ORD-1234,ORDER_ID


### Generate BIO Sequence Tags

BIO tagging means:

- B = Beginning of an entity
- I = Inside an entity
- O = Outside any entity

In [8]:
def create_bio_tags(text, entity_value, entity_type):
    tokens = text.split()
    tags = []

    for token in tokens:
        clean_token = token.strip(".,!?")

        if clean_token == entity_value:
            tags.append(f"B-{entity_type}")
        else:
            tags.append("O")

    return tokens, tags


entity_data[["tokens", "bio_tags"]] = entity_data.apply(
    lambda row: pd.Series(
        create_bio_tags(
            row["text"],
            row["entity_value"],
            row["entity_type"]
        )
    ),
    axis=1
)

entity_data[["text", "tokens", "bio_tags"]]

,text,tokens,bio_tags
0,Please check order ORD-1042 for me.,"[Please, check, order, ORD-1042, for, me.]","[O, O, O, B-ORDER_ID, O, O]"
1,The carrier tracking code TRK-78421 has not ch...,"[The, carrier, tracking, code, TRK-78421, has,...","[O, O, O, O, B-TRACKING_CODE, O, O, O]"
2,Product SKU-AX19 arrived damaged.,"[Product, SKU-AX19, arrived, damaged.]","[O, B-PRODUCT_SKU, O, O]"
3,I need help with order ORD-1234 and tracking c...,"[I, need, help, with, order, ORD-1234, and, tr...","[O, O, O, O, O, B-ORDER_ID, O, O, O, O]"


### Display Sequence Tags Clearly

In [9]:
for _, row in entity_data.iterrows():
    print("Text:", row["text"])
    print("Token-Level Labels:")

    for token, tag in zip(row["tokens"], row["bio_tags"]):
        print(f"{token:10} -> {tag}")

    print("-" * 60)

Text: Please check order ORD-1042 for me.
Token-Level Labels:
Please     -> O
check      -> O
order      -> O
ORD-1042   -> B-ORDER_ID
for        -> O
me.        -> O
------------------------------------------------------------
Text: The carrier tracking code TRK-78421 has not changed.
Token-Level Labels:
The        -> O
carrier    -> O
tracking   -> O
code       -> O
TRK-78421  -> B-TRACKING_CODE
has        -> O
not        -> O
changed.   -> O
------------------------------------------------------------
Text: Product SKU-AX19 arrived damaged.
Token-Level Labels:
Product    -> O
SKU-AX19   -> B-PRODUCT_SKU
arrived    -> O
damaged.   -> O
------------------------------------------------------------
Text: I need help with order ORD-1234 and tracking code TRK-99102.
Token-Level Labels:
I          -> O
need       -> O
help       -> O
with       -> O
order      -> O
ORD-1234   -> B-ORDER_ID
and        -> O
tracking   -> O
code       -> O
TRK-99102. -> O
-------------------------------------

### Export Label-Engineered Datasets

In [10]:
multi_class_dataset = raw_data[[
    "customer_message",
    "engineered_label",
    "label_id"
]]

instruction_response_dataset = instruction_data[[
    "instruction",
    "input",
    "response",
    "prompt",
    "target"
]]

sequence_tagging_dataset = entity_data[[
    "text",
    "tokens",
    "bio_tags"
]]

multi_class_dataset.to_csv("ecommerce_multiclass_labels.csv", index=False)

instruction_response_dataset.to_json(
    "ecommerce_instruction_response.jsonl",
    orient="records",
    lines=True
)

sequence_tagging_dataset.to_json(
    "ecommerce_sequence_tags.jsonl",
    orient="records",
    lines=True
)

print("E-commerce label-engineered datasets exported successfully.")


E-commerce label-engineered datasets exported successfully.
